# Tutorial PXCT data analysis (PART 3)

### Tutor: Julio C. da Silva (Néel Institute CNRS, Grenoble, France) 
### email: julio-cesar.da-silva@neel.cnrs.fr
#### Personal webpage: https://sites.google.com/view/jcesardasilva

### <span style="color:red">** Disclaimer: This notebook is intended from educational reasons only.**</span>
<span style="color:red">**Warning: You should have completed parts 1 and 2 before starting part 3**</span>

<table class="tfo-notebook-buttons" align="center">
  <td>
    <a target="_blank" rel="noopener noreferrer" href="https://github.com/jcesardasilva/toupy"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

#### Importing packages again
Since we start a new notebook, we need to import the packages again:

In [ ]:
# standard packages
import sys
import time
# third party packages
from IPython import display as ipy_display
import matplotlib.pyplot as plt
import numpy as np
import toupy

### Interactive backend

In [ ]:
%matplotlib widget

#### Let us reload our data 
We do this the same way we did in Part 2, but we only change the filename to `PXCTalignedprojections.npz`:

In [ ]:
fname = 'PXCTalignedprojections.npz'
data_dict = np.load(fname) # load the file
list(data_dict.files) # this one list the keys of the data dictionary extracted from the file
wavelen = data_dict['wavelen']
pixsize = data_dict['psize']
theta = data_dict['theta']
projections = data_dict['projections'] # <- ATTENTION: this one is memory consuming. 
nproj, nr, nc = projections.shape
delta_theta = np.diff(np.sort(theta))[0]

print(f"The total number of projections is {nproj}")
print(f"The angular sampling interval is {delta_theta:.02f} degrees")
print(f"The projection pixel size of the projections is {pixsize/1e-9:.02f} nm")
print(f"The wavelenth of the incoming photons is {wavelen/1e-10:.02f} Angstroms")

In [ ]:
plt.close('all')
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3), constrained_layout=True)
ax1.imshow(projections[0], cmap='bone', vmin=-0.15, vmax=0.15)
ax1.set_title('Derivative proj. at 0°', fontsize=13)
ax1.axis('off')
ax2.imshow(projections[-1], cmap='bone', vmin=-0.15, vmax=0.15)
ax2.set_title(f'Derivative proj. at {theta[-1]:.1f}°', fontsize=13)
ax2.axis('off')
plt.show()

### Step 4 — Tomographic reconstruction

We now reconstruct the full 3-D volume using **Filtered Back Projection (FBP)**. FBP is the standard algorithm for parallel-beam CT: each projection is filtered in Fourier space (to compensate for the density of sampling in the sinogram), then back-projected across the reconstruction plane.

Because we work with **derivative projections**, FBP naturally integrates them during back-projection, recovering the original phase values in the reconstructed slices.

> ⚠️ **This step is time and memory intensive.** The full volume reconstruction processes all slices. On a modern laptop with 8 cores it takes a few minutes.
>
> Tip: set `params["showrecons"] = True` to display each reconstructed slice on the fly (slower but useful for checking quality during reconstruction).

### The tomographic reconstruction
Congratulations! You finally arrive to the tomographic reconstruction step of the entire volume.

<span style="color:red">**Warning: This part is time and memory consuming**</span>. This could be optimized by, for example, implementing the reconstruction on a GPU card or distributing the reconstruction of a group of slices accross different CPU cores. Since our goal here is learning, we will stick to the reconstruction using a CPU only.

In [ ]:
# import the toupy routines we will need
from toupy.tomo import full_tomo_recons

params = dict()
params["slicenum"]       = 200      # slice index for the preview reconstruction
params["filtertype"]     = "ram-lak"  # FBP filter: 'ram-lak', 'shepp-logan', 'hann', 'hamming', None
params["freqcutoff"]     = 1.0      # filter frequency cutoff [0–1]; 1.0 = no extra low-pass
params["circle"]         = False    # mask outside the inscribed circle (True can reduce FBP ringing)
params["algorithm"]      = "FBP"    # reconstruction algorithm
params["derivatives"]    = True     # input is derivative projections → FBP integrates during BP
params["calc_derivatives"] = False  # derivatives already computed in Part 2
params["cliplow"]        = None     # display clip on low end (air)
params["cliphigh"]       = -1e-4   # display clip on high end (dense material)
params["autosave"]       = False
params["vmin_plot"]      = None
params["vmax_plot"]      = -1e-4
params["colormap"]       = "bone"
params["showrecons"]     = False    # set True to display slices during reconstruction (slower)

In [ ]:
params = dict() # initializing dictionary
params["slicenum"] = 200  # Choose the slice for the initial reconstruction
params["filtertype"] = "ram-lak" # options: `ram-lak`, `shepp-logan`, `cosine`, `hamming`,`hann`, None (no filter).
params["freqcutoff"] = 1.0  # Frequency cutoff (between 0 and 1)
params["circle"] = False#True 
params["algorithm"] = "FBP"  # FBP or SART
params["derivatives"] = True  # To use the derivatives of the projections
params["calc_derivatives"] = False  # Calculate derivatives if not done
params["cliplow"] = None  # clip air threshold
params["cliphigh"] = -1e-4  # clip on sample threshold
params["autosave"] = False
params["vmin_plot"] = None  # 0.5e-5
params["vmax_plot"] = -1e-4  # None
params["colormap"] = "bone"
params["showrecons"] = False  # to display the reconstructed slice on the fly.

And now, we start the reconstruction:

#### Understanding the reconstructed gray levels

The FBP output is in units of **phase shift per pixel** (rad/px). To convert to the physically meaningful **refractive index decrement δ**, we use:

$$\delta = \frac{\phi \cdot \lambda}{2\pi \cdot \Delta z}$$

where φ is the phase (rad), λ is the X-ray wavelength, and Δz = voxelsize is the voxel size along the beam direction.

Higher δ values correspond to denser / higher-Z regions of the sample (e.g. zeolite crystallites in an FCC particle). Lower δ values correspond to pores (air, δ ≈ 0).

In [ ]:
tomogram = full_tomo_recons(projections, theta, **params)

#### Orthogonal view of the reconstructed volume
Let us display three orthogonal view of the reconstructed volume

In [ ]:
# third packages
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
# import the toupy routines we will need
from toupy.utils import convert_to_delta, tqdm

The gray level of the reconstructed slices are given in units of phase-shifts. Let us now convert phase-shifts to $\delta$ , where $\delta$ is the refractive index decrement and the refractive index is $n=1-\delta+i\beta$.

In [ ]:
nslices, nr, nc = tomogram.shape
energy = 12.4e-10/wavelen
voxelsize = pixsize
print(f"The energy of the incident photons is {energy:.02f} keV")
print(f"The voxel size of the reconstructed volume is {voxelsize/1e-9:.02f} nm")
print(f"The dimensions of the reconstructed volume is {tomogram.shape} (height,width,depth)")

In [ ]:
params["slicenum"] = 150
params["vmin_plot"] = 2e-6  # None
params["vmax_plot"] = 2e-5  # 5e-4
params["scale_bar_size"] = 2  # in microns
params["scale_bar_height"] = 0.4
params["scale_bar_color"] = "yellow"
params["bar_start"] = [20, 70]
params["bar_axial"] = [70, 100]  # [cols,rows]
params["colormap"] = "bone"
params["interpolation"] = "nearest"

Conversion of the gray-level of the slices from phase-shifts to delta:

In [ ]:
# loop over the tomogram
for ii in tqdm(range(nslices), desc="Converting slices", colour='blue', file=sys.stdout):
    tomogram[ii], factor = convert_to_delta(tomogram[ii], energy, voxelsize)

Preparing the Sagital, Coronal, and Axial slices:

In [ ]:
%matplotlib inline
# Sagital slice (XZ plane — view from the side)
textstr = r"{} $\mu$m".format(params["scale_bar_size"])

figsag, axsag = plt.subplots(figsize=(12, 5), constrained_layout=True)
imsag = axsag.imshow(
    sagital_slice,
    interpolation=params['interpolation'],
    cmap=params["colormap"],
    vmin=params["vmin_plot"],
    vmax=params["vmax_plot"],
)
figsag.colorbar(imsag, label='δ (refractive index decrement)')
axsag.set_title("Sagital slice (XZ plane)")
axsag.text(
    params["bar_start"][0] + 2, params["bar_start"][1] - 5,
    textstr, fontsize=14, verticalalignment="bottom", color=params["scale_bar_color"],
)
import matplotlib.patches as patches
axsag.add_patch(patches.Rectangle(
    (params["bar_start"][0], params["bar_start"][1]),
    np.round(params["scale_bar_size"] * 1e-6 / voxelsize),
    np.round(params["scale_bar_height"] * 1e-6 / voxelsize),
    color=params["scale_bar_color"],
))
axsag.set_axis_off()
figsag.savefig('sagital_recons.png', dpi=150)
plt.show()
print("Saved sagital_recons.png")

In [ ]:
# Coronal slice (YZ plane — view from the front)
figcor, axcor = plt.subplots(figsize=(12, 5), constrained_layout=True)
imcor = axcor.imshow(
    coronal_slice,
    interpolation=params['interpolation'],
    cmap=params["colormap"],
    vmin=params["vmin_plot"],
    vmax=params["vmax_plot"],
)
figcor.colorbar(imcor, label='δ (refractive index decrement)')
axcor.set_title("Coronal slice (YZ plane)")
axcor.text(
    params["bar_start"][0] + 2, params["bar_start"][1] - 5,
    textstr, fontsize=14, verticalalignment="bottom", color=params["scale_bar_color"],
)
axcor.add_patch(patches.Rectangle(
    (params["bar_start"][0], params["bar_start"][1]),
    np.round(params["scale_bar_size"] * 1e-6 / voxelsize),
    np.round(params["scale_bar_height"] * 1e-6 / voxelsize),
    color=params["scale_bar_color"],
))
axcor.set_axis_off()
figcor.savefig('coronal_recons.png', dpi=150)
plt.show()
print("Saved coronal_recons.png")

In [ ]:
# Axial slice (XY plane — top-down view, slice at params["slicenum"])
figaxial, axaxial = plt.subplots(constrained_layout=True)
crop = axial_slice[60:-40, 40:-60]
imaxial = axaxial.imshow(
    crop,
    interpolation=params['interpolation'],
    cmap=params["colormap"],
    vmin=params["vmin_plot"],
    vmax=params["vmax_plot"],
)
axaxial.text(
    params["bar_axial"][0] + 2 - 48, params["bar_axial"][1] - 5,
    textstr, fontsize=16, verticalalignment="bottom", color=params["scale_bar_color"],
)
figaxial.colorbar(imaxial, label='δ (refractive index decrement)')
axaxial.add_patch(patches.Rectangle(
    (params["bar_axial"][0] - 50, params["bar_axial"][1]),
    np.round(params["scale_bar_size"] * 1e-6 / voxelsize),
    np.round(params["scale_bar_height"] * 1e-6 / voxelsize),
    color=params["scale_bar_color"],
))
axaxial.set_axis_off()
plt.show()

In [ ]:
###### Coronal slice
figcor = plt.figure(num=2, figsize=(12,5), constrained_layout=True)
axcor = figcor.add_subplot(111)
imcor = axcor.imshow(
    coronal_slice,
    interpolation=params['interpolation'],
    cmap=params["colormap"],
    vmin=params["vmin_plot"],
    vmax=params["vmax_plot"],
)
figcor.colorbar(imcor)
axcor.set_title("Coronal slice")
axcor.text(
    params["bar_start"][0] + 2,
    params["bar_start"][1] - 5,
    textstr,
    fontsize=14,
    verticalalignment="bottom",
    color=params["scale_bar_color"],
)
rectcor = patches.Rectangle(
    (params["bar_start"][0], params["bar_start"][1]),  # (x,y)
    (np.round(params["scale_bar_size"] * 1e-6 / voxelsize)),  # width
    (np.round(params["scale_bar_height"] * 1e-6 / voxelsize)),  # height
    color=params["scale_bar_color"],
)
axcor.add_patch(rectcor)
axcor.set_axis_off()
#figcor.savefig('coronal_recons.png')

In [ ]:
figaxial = plt.figure(num=3, constrained_layout=True)#, figsize=(12,8))
axaxial = figaxial.add_subplot(111)
imaxial = axaxial.imshow(
    axial_slice[60:-40,40:-60],
    interpolation=params['interpolation'],
    cmap=params["colormap"],
    vmin=params["vmin_plot"],
    vmax=params["vmax_plot"])
axaxial.text(
    params["bar_axial"][0] + 2-48,
    params["bar_axial"][1] - 5,
    textstr,
    fontsize=16,
    verticalalignment="bottom",
    color=params["scale_bar_color"],
)
figaxial.colorbar(imaxial)
rectaxial = patches.Rectangle(
    (params["bar_axial"][0]-50, params["bar_axial"][1]),  # (x,y)
    (np.round(params["scale_bar_size"] * 1e-6 / voxelsize)),  # width
    (np.round(params["scale_bar_height"] * 1e-6 / voxelsize)),  # height
    color=params["scale_bar_color"],
)
axaxial.add_patch(rectaxial)
axaxial.set_axis_off()
#figaxial.savefig('axial_recons.png')

In [ ]:
outputfname = "PXCTtomogram.npz"
np.savez(outputfname, wavelen = wavelen, energy=energy, voxelsize = voxelsize, tomogram_delta = tomogram)

In [ ]:
!ls -lrth